# 05 · Evaluator–Optimizer with LangChain

Closed-loop quality control. One LLM **generates**, another **scores** it against criteria, and we **loop** — feeding the feedback back into the generator — until it passes a threshold or we hit a max-iteration cap.

```
brief ─► [generate] ─► [evaluate] ──pass?──► done
            ▲                        │no
            └────── feedback ────────┘
```

---

## Setup

In [ ]:
%pip install -qU langchain langchain-openai python-dotenv

In [1]:
import os
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../../.env")

# A touch of creativity for the generator; the evaluator stays strict (temp 0).
generator_llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0.8)
evaluator_llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0.0)
parser = StrOutputParser()

---
## 1. The generator

It writes a first attempt, and on later rounds **revises** using the evaluator's feedback.

In [2]:
generator = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a senior copywriter. Write ONE product tagline (max 8 words). "
                   "If a previous attempt and feedback are given, revise to address the feedback."),
        ("human", "Brief: {brief}\n\nPrevious attempt: {previous}\nFeedback: {feedback}"),
    ])
    | generator_llm | parser
)

---
## 2. The evaluator

Structured output makes the score **machine-readable** — that's what lets us branch on it. A `passed` boolean plus a numeric `score` and actionable `feedback`.

In [5]:
class Evaluation(BaseModel):
    score: int = Field(description="Quality score from 1 (poor) to 10 (excellent)", ge=1, le=10)
    passed: bool = Field(description="True only if the tagline clearly meets the bar")
    feedback: str = Field(description="One or two sentences of specific, actionable feedback")

evaluator = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a strict brand editor. Score the tagline 1-10 on punchiness, clarity, "
                   "and memorability. Set passed=true ONLY if score >= 8. Be demanding."),
        ("human", "Brief: {brief}\nTagline: {tagline}"),
    ])
    | evaluator_llm.with_structured_output(Evaluation)
)

---
## 3. The loop

Two stopping conditions — **quality met** OR **max iterations** — so the loop can never run forever (and your token bill stays bounded).

In [14]:
def optimize(brief: str, threshold: int = 9, max_iters: int = 4) -> str:
    previous, feedback = "(none)", "(none)"
    tagline = ""
    for i in range(1, max_iters + 1):
        tagline = generator.invoke({"brief": brief, "previous": previous, "feedback": feedback})
        ev = evaluator.invoke({"brief": brief, "tagline": tagline})
        print(f"Iter {i}: score={ev.score:>2}  {tagline!r}")
        print(f"         feedback: {ev.feedback}")
        if ev.passed or ev.score >= threshold:
            print(f"\n✅ Passed on iteration {i}, score={ev.score}, returning...")
            return tagline
        previous, feedback = tagline, ev.feedback
    print(f"\n⚠️  Stopped at max_iters={max_iters}; returning best-effort last attempt")
    return tagline

In [15]:
final = optimize(
    "A developer tool that auto-generates API docs from your code. "
    "Audience: backend engineers. Tone: confident, a little witty."
)
print("\nFINAL:", final)

Iter 1: score= 8  'Code Your Docs, Effortlessly'
         feedback: The tagline is clear and concise, effectively communicating ease of use with a touch of wit. To improve memorability, consider a more distinctive phrase or rhyme.

✅ Passed on iteration 1, score=8, returning...

FINAL: Code Your Docs, Effortlessly


---
## 4. Watch the cost

Every iteration is **two** LLM calls (generate + evaluate). The loop is powerful but not free — always cap `max_iters`, and consider a cheaper model for the generator vs. a stronger one for the evaluator (or vice-versa).

In [16]:
# Tighten the bar to force more rounds and see the loop work harder.
_ = optimize("A minimalist to-do app for people who hate to-do apps.", threshold=9, max_iters=5)

Iter 1: score= 7  'Simplify Your Day, Skip the Clutter'
         feedback: The tagline is clear and somewhat memorable, but it lacks punchiness and could be more distinctive to stand out in the crowded app market.
Iter 2: score= 8  'Your Day, No Fuss, Just Done'
         feedback: The tagline is clear and concise, effectively communicating simplicity and ease of use. To improve memorability, consider a more rhythmic or catchy phrase.

✅ Passed on iteration 2, score=8, returning...


---
## What to remember

| Piece | Role |
|---|---|
| Generator | Produces output; **revises** when handed feedback |
| Evaluator + `with_structured_output` | Returns a machine-readable `score` / `passed` / `feedback` |
| Stopping conditions | `passed`/threshold **and** `max_iters` — never loop forever |
| Two-model split | Creative generator (high temp) vs. strict evaluator (temp 0) |

**vs. orchestrator–worker (04):** that plans *once* and executes; this *re-generates* until a quality bar is met. Different axes — you can even combine them (evaluate each worker's section).

**Next:** 06 — **Agents & tool calling**: let the model decide its own next action and call real tools.